# Y6 test projection: nside, bins, `shearm` and n(z)

Tomasz's DES Y6 UFalcon test projection (`000_test`), compared with the Y6 source redshift realization set.
A full scan of all 50000 files (2026-11-18, outside this notebook) found one `shearm` set and one n(z) in every file,
equal to row 0 of `desy6_source_redshift.h5`. This notebook shows it for a handful of files.

Needs `/capstor/store/cscs/userlab/lp184` inside the container.

In [ ]:
from pathlib import Path

import h5py
import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

root = Path("/capstor/store/cscs/userlab/lp184/tomaszk/261109_cosmogridv1_desy6")
grid_dir = root / "000_test/cosmogrid/processed/desy6/CosmoGrid/bary/grid"
nz_file = root / "data_desy6a2/desy6_source_redshift.h5"
fname = "projected_probes_maps_v11dmb.h5"

src_bins = [f"metacal{i}" for i in range(1, 5)]
lens_bins = [f"maglim{i}" for i in range(1, 5)]

In [ ]:
# three random cosmologies, two permutations each
rng = np.random.default_rng(1)
cosmo_dirs = sorted(grid_dir.glob("cosmo_*"))
files = []
for i in rng.choice(len(cosmo_dirs), size=3, replace=False):
    perm_dirs = sorted(cosmo_dirs[i].glob("perm_*"))
    files += [perm_dirs[j] / fname for j in rng.choice(len(perm_dirs), size=2, replace=False)]

print(f"{len(cosmo_dirs)} cosmologies")
for path in files:
    print(path.relative_to(grid_dir))

## Map resolution and bins

In [ ]:
with h5py.File(files[0], "r") as f:
    for probe in f["map"]:
        for b in f["map"][probe]:
            npix = f["map"][probe][b].shape[0]
            print(f"map/{probe}/{b:9s} npix = {npix:>9d}  nside = {hp.npix2nside(npix)}")

## `shearm` and n(z) per file

In [ ]:
def read_nuisances(path):
    with h5py.File(path, "r") as f:
        m = np.array([f["nz"][b].attrs["shearm"] for b in src_bins + lens_bins])
        nz = np.stack([f["nz"][b][:] for b in src_bins])  # (4, 300, 2), columns z and n(z)
    return m, nz


m_files, nz_files = map(np.stack, zip(*[read_nuisances(p) for p in files]))

print(f"{'file':24s}" + "".join(f"{b:>10s}" for b in src_bins + lens_bins))
for path, m in zip(files, m_files):
    name = f"{path.parts[-3]}/{path.parts[-2]}"
    print(f"{name:24s}" + "".join(f"{x:+10.5f}" for x in m))
print()
print("same shearm in every file:", np.all(m_files == m_files[0]))
print("same n(z) in every file:  ", np.all(nz_files == nz_files[0]))

## Against the realization set

`desy6_source_redshift.h5` holds 80760 joint (n(z), m) realizations. Which one do the files carry?

In [ ]:
n_show = 2000  # n(z) realizations loaded for the band below
with h5py.File(nz_file, "r") as f:
    m_set = np.stack([f[f"M_BIN{i}"][:] for i in range(1, 5)], axis=1)  # (80760, 4)
    nz_set = np.stack([f[f"NZ_BIN{i}"][:n_show] for i in range(1, 5)], axis=1)  # (n_show, 4, 300)
    z_mid = f["Z_MID"][:]

rows = np.flatnonzero(np.all(np.isclose(m_set, m_files[0, :4], rtol=0, atol=1e-12), axis=1))
print(f"rows of M_BIN1-4 equal to the files' shearm: {rows} (of {len(m_set)})")
print("file z grid == Z_MID:     ", np.all(nz_files[0, :, :, 0] == z_mid))
print("file n(z) == NZ_BIN row 0:", np.allclose(nz_files[0, :, :, 1], nz_set[0], rtol=1e-6, atol=0))
print()
for i, b in enumerate(src_bins):
    z = (m_files[0, i] - m_set[:, i].mean()) / m_set[:, i].std()
    print(f"{b}: shearm {m_files[0, i]:+.5f}, set {m_set[:, i].mean():+.5f} +- {m_set[:, i].std():.5f} ({z:+.2f} sigma)")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6), constrained_layout=True)
for i, b in enumerate(src_bins):
    ax = axes[0, i]
    ax.hist(m_set[:, i], bins=80, color="0.7", label="realization set")
    for k, m in enumerate(m_files[:, i]):
        ax.axvline(m, color="C3", lw=1, label=f"{len(files)} files (overlap)" if k == 0 else None)
    ax.set(title=b, xlabel="m")

    ax = axes[1, i]
    lo, hi = np.percentile(nz_set[:, i], [2.5, 97.5], axis=0)
    ax.fill_between(z_mid, lo, hi, color="0.8", label=f"95% of first {n_show} realizations")
    for k, nz in enumerate(nz_files[:, i]):
        ax.plot(nz[:, 0], nz[:, 1], color="C3", lw=1, label=f"{len(files)} files (overlap)" if k == 0 else None)
    ax.set(xlabel="z", xlim=(0, 2.5))

axes[0, 0].set_ylabel("realizations")
axes[1, 0].set_ylabel("n(z)")
axes[0, 0].legend(fontsize=8)
axes[1, 0].legend(fontsize=8)
plt.show()